In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("C:\\Users\\ASUS\\Desktop\\Narmada Project\\Data\\Raw\\Narmada.csv")

df.head()

In [ ]:
df.head()
df.info()

In [ ]:
#Missing value analysis
missing = df.isnull().sum()
missing_percent = (missing/len(df))*100
print(missing)
print(missing_percent)

In [ ]:
# Level gaps
level_groups = (df['level'].notna() != df['level'].notna().shift()).cumsum()

level_gap_lengths = (
    df[df['level'].isna()]
    .groupby(level_groups)
    .size()
    .sort_values(ascending=False)
)

print("Largest level gaps:")
print(level_gap_lengths.head(20))


# Flow gaps
flow_groups = (df['flow'].notna() != df['flow'].notna().shift()).cumsum()

flow_gap_lengths = (
    df[df['flow'].isna()]
    .groupby(flow_groups)
    .size()
    .sort_values(ascending=False)
)

print("\nLargest flow gaps:")
print(flow_gap_lengths.head(20))

In [ ]:
### gives no. of unique coordinated

print(df['lat_x'].nunique())
print(df['lon_x'].nunique())

print(df[['lat_x','lon_x']].drop_duplicates())

In [ ]:
for station, group in df.groupby(['lat_x','lon_x']):
    print(station, len(group))

In [ ]:
for station, group in df.groupby(['lat_x','lon_x']):
    print("\nStation:", station)
    print("Start:", group['Date'].min())
    print("End  :", group['Date'].max())

In [ ]:
## returns how many null values of level and flow each station has

df.groupby(['lat_x','lon_x'])[['level','flow']].apply(
    lambda x: x.isnull().sum()
)

In [ ]:
print(df[['level','flow']].corr())

In [ ]:
df['station_id'] = (
    df.groupby(['lat_x','lon_x'])
      .ngroup()
)

df[['lat_x','lon_x','station_id']].drop_duplicates()

In [ ]:
print("Rows with both targets available:")
print(df[['level','flow']].dropna().shape[0])

print("\nTotal rows:")
print(len(df))

In [ ]:
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df_model = df.dropna(subset=['level', 'flow']).copy()

'''pd.to_datetime("01/02/2010")  will o/p => Timestamp('2010-01-02 00:00:00')'''


'''Missing values were present in the target variables level and flow. Since several missing
 segments extended over hundreds of consecutive observations, 
 interpolation was avoided to prevent introducing artificial hydrological behavior.
 Rows with unavailable target values were excluded from model development, resulting in 32,943 valid observations.'''

In [ ]:
# For each station, calculate the gaps between consecutive dates.(ex. in station1 the i day gap i.e. the data is collected for each day for 6547 days)
# Then count how often each gap (in days) occurs, and show the top 5.

for station, group in df_model.groupby('station_id'):
        print(f"\nStation {station}")
        print(group['Date'].diff().value_counts().head())   

In [ ]:
# Loop through the DataFrame, grouping rows by each station_id
for station, group in df.groupby('station_id'):
    # Print the station number to label the output
    print(f"\nStation {station}")

    # Create a Boolean Series: True if both 'level' and 'flow' are not missing (valid data), False otherwise
    both_available = group[['level','flow']].notna().all(axis=1)

    # Group consecutive True/False runs together using cumsum on changes,
    # then sum each group to get the length of continuous valid segments
    segments = both_available.groupby(
        (both_available != both_available.shift()).cumsum()
    ).sum()

    # Print the longest continuous run of valid (non-missing) data for this station
    print("Longest continuous valid segment:",
          segments.max())


In [ ]:
level_missing = set(df[df['level'].isna()].index)
flow_missing = set(df[df['flow'].isna()].index)

print("Common missing:", len(level_missing & flow_missing))
print("Level only:", len(level_missing - flow_missing))
print("Flow only:", len(flow_missing - level_missing))

In [ ]:
##find the distribution of segment lengths.

df_temp = df.copy()

df_temp['valid'] = (
    df_temp['level'].notna() &
    df_temp['flow'].notna()
)

df_temp['segment_id'] = (
    (df_temp['valid'] != df_temp['valid'].shift())
    .cumsum()
)

segment_lengths = (
    df_temp[df_temp['valid']]
    .groupby(['station_id','segment_id'])
    .size()
)

print(segment_lengths.describe())

print(segment_lengths.sort_values(ascending=False).head(20))

In [ ]:
#Vaild continuous data with a threshold of 900 days

# valid rows
df['valid'] = (
    df['level'].notna() &
    df['flow'].notna()
)

# segment ids
df['segment_id'] = (
    (df['valid'] != df['valid'].shift())
    .cumsum()
)

# segment lengths
segment_lengths = (
    df[df['valid']]
    .groupby(['station_id','segment_id'])
    .size()
    .reset_index(name='length')
)

# keep only segments >= 90 days
valid_segments = segment_lengths[
    segment_lengths['length'] >= 90
][['station_id','segment_id']]

df_filtered = df.merge(
    valid_segments,
    on=['station_id','segment_id'],
    how='inner'
)

print(df_filtered.shape)

In [ ]:
df_filtered[['level','flow']].isnull().sum()

In [ ]:

df_filtered['month'] = df_filtered['Date'].dt.month
df_filtered['dayofyear'] = df_filtered['Date'].dt.dayofyear

df_filtered['month_sin'] = np.sin(2*np.pi*df_filtered['month']/12)
df_filtered['month_cos'] = np.cos(2*np.pi*df_filtered['month']/12)

df_filtered['doy_sin'] = np.sin(2*np.pi*df_filtered['dayofyear']/365)
df_filtered['doy_cos'] = np.cos(2*np.pi*df_filtered['dayofyear']/365)

df_filtered.columns

In [ ]:
FEATURES = [
    'station_id',
    'Rain',
    'Tmin',
    'Tmax',
    'WS50M',
    'T2M',
    'RH2M',
    'PS',
    'WS2M',
    'GWETTOP',
    'month_sin',
    'month_cos',
    'doy_sin',
    'doy_cos',
    'level',
    'flow'
]

df_model = df_filtered[FEATURES].copy()
print(df_model.columns.tolist())

In [ ]:
train_ratio = 0.70
val_ratio = 0.15

n = len(df_filtered)

train_end = int(n * train_ratio)
val_end = int(n * (train_ratio + val_ratio))

train_df = df_model.iloc[:train_end]
val_df = df_model.iloc[train_end:val_end]
test_df = df_model.iloc[val_end:]

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

In [ ]:
print(len(train_df)/len(df_model))
print(len(val_df)/len(df_model))
print(len(test_df)/len(df_model))

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

train_scaled = scaler.fit_transform(train_df)
val_scaled = scaler.transform(val_df)
test_scaled = scaler.transform(test_df)

import joblib

joblib.dump(scaler, "scaler.pkl")

In [ ]:
print(train_scaled.min(), train_scaled.max())
print(val_scaled.min(), val_scaled.max())
print(test_scaled.min(), test_scaled.max())

In [ ]:
df_filtered.head()
df_filtered[['station_id','segment_id']].tail()

In [ ]:
print(df_filtered[['station_id','segment_id']].iloc[22740:22745])
print(df_filtered[['station_id','segment_id']].iloc[27610:27615])

In [ ]:
### SEQUENCE GENERATION

WINDOW = 30

target_cols = ['level', 'flow']

target_idx = [
    FEATURES.index(col)
    for col in target_cols
]

In [ ]:
def create_sequences(data, window, target_idx):

    X = []
    y = []

    for i in range(len(data) - window):

        X.append(
            data[i:i+window]
        )

        y.append(
            data[i+window, target_idx]
        )

    return np.array(X), np.array(y)

In [ ]:
X_train, y_train = create_sequences(
    train_scaled,
    WINDOW,
    target_idx
)

X_val, y_val = create_sequences(
    val_scaled,
    WINDOW,
    target_idx
)

X_test, y_test = create_sequences(
    test_scaled,
    WINDOW,
    target_idx
)

In [ ]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

In [ ]:
np.save("X_train.npy", X_train)
np.save("y_train.npy", y_train)

np.save("X_val.npy", X_val)
np.save("y_val.npy", y_val)

np.save("X_test.npy", X_test)
np.save("y_test.npy", y_test)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

target_scaler = MinMaxScaler()

target_scaler.fit(
    train_df[['level','flow']]
)

joblib.dump(
    target_scaler,
    "target_scaler.pkl"
)